# Experiment 1: Fuzzy C-Means from scratch on a small test image

First, non-vectorised implementation of Fuzzy C-Means (loop over pixels) tested on a small 159x318 satellite crop (k = 4, m = 1.7). **Preliminary experiment**: the final, vectorised and evaluated version is in `notebooks/fuzzy_c_means_satellite_segmentation.ipynb`. The test image is not redistributed: put your own image in `../data/TESTsat.jpg`.

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
# Lire l'image en couleur (par défaut, cv2 lit en BGR)
image_color = cv2.imread("../data/TESTsat.jpg")

# Vérification
if image_color is None:
    print("Erreur : L'image n'a pas été trouvée. Vérifie le nom ou le chemin.")
else:
    # Convertir de BGR à RGB pour affichage correct avec matplotlib
    image_rgb = cv2.cvtColor(image_color, cv2.COLOR_BGR2RGB)

    # Afficher l'image
    plt.figure(figsize=(8, 8))
    plt.imshow(image_rgb)
    plt.title("Image Satellite Originale")
    plt.axis('off')
    plt.show()

In [ ]:
# Chargement de l'image (TESTsat.jpg) en niveau de gris
image_gray = cv2.imread("../data/TESTsat.jpg", cv2.IMREAD_GRAYSCALE)

# Vérification
if image_gray is None:
    print("Erreur : L'image n'a pas été trouvée. Vérifie le chemin ou le nom.")
else:
    # Affichage de l'image
    plt.figure(figsize=(8, 8))
    plt.imshow(image_gray, cmap='gray')
    plt.title("Image Satellite en niveaux de gris")
    plt.axis('off')
    plt.show()

    # Dimensions de l'image
    print("Dimensions de l'image :", image_gray.shape)

Dimensions de l'image : (159, 318)


In [ ]:
# Normalisation : met les valeurs de pixels entre 0 et 1
image_flat = image_gray / 255.0

# Dimensions originales de l'image (hauteur, largeur)
original_shape = image_flat.shape

# Aplatir l'image : transforme l'image 2D (H x W) en un vecteur 1D (H*W,)
pixels = image_flat.flatten()

# Vérification
print("Forme de l'image après aplatissage :", pixels.shape)
print("Valeurs min/max après normalisation :", pixels.min(), "/", pixels.max())

Forme de l'image après aplatissage : (50562,)
Valeurs min/max après normalisation : 0.0 / 1.0


In [ ]:
# Paramètres FCM
k = 4              # Nombre de clusters (à ajuster selon ton image)
m = 1.7            # Paramètre de flou (>1)
n = len(image_flat) # Nombre total de pixels (données)

# Fonction pour initialiser la matrice d'appartenance U
def initialize_membership_matrix(n, k):
    membership_mat = []
    for _ in range(n):
        random_values = np.random.dirichlet(np.ones(k), size=1)[0]
        membership_mat.append(random_values)
    return np.array(membership_mat)

# Initialisation
U = initialize_membership_matrix(n, k)

# Vérification
print("Matrice d'appartenance initialisée de forme :", U.shape)
print("Exemple (1er pixel) :", U[0])
print("Somme :", np.sum(U[0])) #elle doit etre egale a 1

Matrice d'appartenance initialisée de forme : (159, 4)
Exemple (1er pixel) : [0.0892685  0.13193445 0.43216742 0.34662963]
Somme : 0.9999999999999998


In [ ]:
# Fonction pour initialiser la matrice d'appartenance U
def initialize_membership_matrix(n, k):
    membership_mat = []
    for _ in range(n):
        random_values = np.random.dirichlet(np.ones(k), size=1)[0]
        membership_mat.append(random_values)
    return np.array(membership_mat)

# Fonction pour mettre à jour les centres des clusters
def update_cluster_centers(U, pixels, m):
    k = U.shape[1]  # Nombre de clusters
    centers = np.zeros(k)  # Initialisation des centres des clusters
    
    for j in range(k):
        num = np.sum((U[:, j] ** m) * pixels)  # Numérateur
        den = np.sum(U[:, j] ** m)  # Dénominateur
        centers[j] = num / den  # Calcul du centre du cluster
    
    return centers

# Fonction pour mettre à jour la matrice d'appartenance U
def update_membership_matrix(pixels, centers, m):
    n = len(pixels)
    k = len(centers)
    U_new = np.zeros((n, k))
    
    # Calcul des distances entre chaque pixel et chaque centre
    for i in range(n):
        distances = np.array([abs(pixels[i] - center) for center in centers])
        
        # Mettre à jour les valeurs d'appartenance pour chaque cluster
        for j in range(k):
            sum_d = np.sum((distances[j] / distances) ** (2 / (m - 1)))
            U_new[i, j] = 1 / sum_d
    
    return U_new

# Algorithme FCM
def fuzzy_cmeans(pixels, k, m, max_iter=100, tol=1e-5):
    n = len(pixels)
    
    # Initialisation de la matrice d'appartenance
    U = initialize_membership_matrix(n, k)
    
    # Boucle FCM
    for iteration in range(max_iter):
        # Mettre à jour les centres des clusters
        centers = update_cluster_centers(U, pixels, m)
        
        # Mettre à jour la matrice d'appartenance
        U_new = update_membership_matrix(pixels, centers, m)
        
        # Vérifier la convergence
        if np.linalg.norm(U_new - U) < tol:
            print(f"Convergence atteinte après {iteration + 1} itérations.")
            break
        
        U = U_new  # Mise à jour de la matrice d'appartenance
        
    return U, centers

# Appliquer l'algorithme FCM
U_final, centers_final = fuzzy_cmeans(pixels, k, m)

# Afficher les résultats
print("Centres des clusters finaux :")
print(centers_final)

Centres des clusters finaux :
[0.29544987 0.45462602 0.74082161 0.12479397]


In [ ]:
# Affecter chaque pixel au cluster ayant l'appartenance la plus élevée
cluster_labels = np.argmax(U_final, axis=1)

# Reshaper les labels des clusters pour les mettre à la taille de l'image d'origine
cluster_labels_reshaped = cluster_labels.reshape(image_gray.shape)

# Visualiser les différentes zones (clusters)
plt.figure(figsize=(8, 8))
plt.imshow(cluster_labels_reshaped, cmap='tab10')  # Utilisation d'un colormap pour les clusters
plt.title("Segmented Image with FCM Clusters")
plt.axis('off')
plt.show()

# Visualiser les centres des clusters
print("Centres des clusters :")
for i, center in enumerate(centers_final):
    print(f"Cluster {i+1} centre : {center}")

Centres des clusters :
Cluster 1 centre : 0.29544987050557026
Cluster 2 centre : 0.4546260201495552
Cluster 3 centre : 0.7408216126031643
Cluster 4 centre : 0.12479397245647725


In [ ]:
plt.figure(figsize=(14, 6))

# 1. Image originale
plt.subplot(1, 2, 1)
plt.imshow(image_rgb)
plt.title("Image Satellite Originale (RGB)")
plt.axis('off')

# 2. Image segmentée (labels reshaped)
plt.subplot(1, 2, 2)
plt.imshow(cluster_labels_reshaped, cmap='tab10')
plt.title(f"Image Segmentée par FCM (k = {k})")
plt.axis('off')

plt.tight_layout()
plt.show()